In [ ]:
import os
import multiprocessing
import jax
import jax.numpy as jnp
import numpy as np
import io
import time
from ipywidgets import widgets, Layout, VBox, HBox, HTML as WidgetHTML
from IPython.display import display, clear_output
import bqplot as bq

# --- 1. The Sovereign Kill Switch ---
shutdown_logic = widgets.HTML(value="""
<script>
    window.kill_session = function() {
        const url = window.location.origin + window.location.pathname.replace('/voila/render/', '/voila/api/shutdown/');
        fetch(url, { method: 'POST', keepalive: true }).then(() => {
            alert("Leviathan: Kernel Terminated. Closing tab.");
            window.close();
        });
        if (window.IPython && IPython.notebook && IPython.notebook.kernel) {
            IPython.notebook.kernel.execute('import os; os._exit(0)');
        }
    };
</script>
""")

# --- 2. The Engine Room (YOUR EXACT CODE) ---
# We assume the code you pasted is saved in 'hmcSampler.py' or available in the path.
# If running in a notebook, ensure the class is defined in a previous cell.
from hmcSampler import HMCSampler 

# --- Tactical Potential Parser ---
# Helper to create U from string for the dashboard
def create_jax_u(expr_str, dims):
    jax_ns = {k: getattr(jnp, k) for k in dir(jnp) if not k.startswith('_')}
    jax_ns['jnp'] = jnp
    def U(q):
        # Your sampler expects q to be (n_params,)
        variables = {'x': q[0], 'y': q[1] if dims >= 2 else 0.0}
        return eval(expr_str, {"__builtins__": None}, {**jax_ns, **variables})
    return U

# --- UI Styling & Components ---
COL_STYLE = Layout(width='33%', border='1px solid #dcdcdc', padding='15px', margin='5px', border_radius='8px')
SLIDER_LAYOUT = Layout(width='95%')
HDR = "<div style='background:#f0f0f0; text-align:center; font-weight:bold; padding:5px; border-radius:5px;'>{text}</div>"

p_text = widgets.Textarea(value='0.5*(x**2 + y**2) + jnp.cos(3*x)', layout=Layout(width='98%', height='100px'))
d_drop = widgets.Dropdown(options=[('2D (x,y)', 2)], value=2, description='Dims:', layout=SLIDER_LAYOUT)

# Dashboard controls mapped to your class attributes
w_sld = widgets.IntSlider(value=16, min=1, max=64, step=1, description='Walkers:', layout=SLIDER_LAYOUT) # Divisible by devices usually
s_sld = widgets.IntSlider(value=500, min=100, max=2000, step=100, description='Samples:', layout=SLIDER_LAYOUT)
b_sld = widgets.IntSlider(value=100, min=0, max=500, step=50, description='Burn-in:', layout=SLIDER_LAYOUT)
sz_sld = widgets.FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description='LF Length:', layout=SLIDER_LAYOUT) # maps to lf_length
st_sld = widgets.IntSlider(value=20, min=5, max=100, description='LF Steps:', layout=SLIDER_LAYOUT) # maps to steps

orb_cb = widgets.Checkbox(value=True, description='Show Animation', indent=False)
trail_sld = widgets.IntSlider(value=10, min=2, max=50, step=1, description='Trail Len:', layout=SLIDER_LAYOUT) 
speed_sld = widgets.IntSlider(value=20, min=1, max=100, description='Speed (ms):', layout=SLIDER_LAYOUT)

run_btn = widgets.Button(description="Execute Campaign", button_style='danger', layout=Layout(width='32%', height='40px'))
clear_btn = widgets.Button(description="Clear History", button_style='warning', layout=Layout(width='32%', height='40px'))
kill_btn_html = widgets.HTML(value="""
<button onclick="kill_session()" 
    style="width:100%; height:40px; background-color:#2c3e50; color:white; font-weight:bold; border:none; border-radius:4px; cursor:pointer;">
    Terminate Session
</button>
""", layout=Layout(width='32%'))

output_area = widgets.Output()

# --- 3. The Visualization Engine (BQPlot) ---
def run_fast_animation(sampler, U_func, trail_len_param):
    # 1. VALIDATION & DATA RESHAPING
    if sampler.orbits is None:
        print("Strategy: No orbits found. Did you check 'Show Animation'?")
        return

    raw_orbits = np.array(sampler.orbits, dtype=np.float32)
    n_w = sampler.n_walkers
    
    try:
        # Reshape logic to handle (Samples, Steps, Walkers) flattening
        n_c = raw_orbits.shape[0] // n_w
        n_steps = raw_orbits.shape[1]
        n_dims = raw_orbits.shape[2]
        
        # Unravel: (Samples, Walkers, Steps, Dims)
        structured = raw_orbits.reshape(n_c, n_w, n_steps, n_dims)
        # Transpose: (Samples, Steps, Walkers, Dims)
        ordered = structured.transpose(0, 2, 1, 3)
        # Flatten Time: (Time, Walkers, Dims)
        anim_data = ordered[:, :-1, :, :].reshape(-1, n_w, n_dims)
    except:
        anim_data = raw_orbits[:, 0, :].reshape(-1, 1, raw_orbits.shape[-1])
        n_w = 1

    total_frames = anim_data.shape[0]

    # 2. SCALES (The Backbone)
    # Global Limits
    LIMIT = 4
    sc_x = bq.LinearScale(min=-LIMIT, max=LIMIT)
    sc_y = bq.LinearScale(min=-LIMIT, max=LIMIT)
    
    # Independent Density Scales (So one histogram doesn't squash the other)
    sc_dens_x = bq.LinearScale() 
    # REVERSE the Y-density so bars grow Right-to-Left (Inwards)
    sc_dens_y = bq.LinearScale(reverse=True) 
    
    sc_color = bq.ColorScale(scheme='Greys', reverse=True) 

    # 3. BACKGROUND POTENTIAL
    x_grid = np.linspace(-LIMIT, LIMIT, 100).astype(np.float32)
    y_grid = np.linspace(-LIMIT, LIMIT, 100).astype(np.float32)
    Z = np.zeros((100, 100), dtype=np.float32)
    
    for i in range(100):
        for j in range(100):
            val = U_func(jnp.array([x_grid[j], y_grid[i]]))
            Z[i, j] = float(val)
    Z = np.nan_to_num(Z, nan=0.0, posinf=100.0, neginf=-100.0)

    heatmap = bq.GridHeatMap(column=x_grid, row=y_grid, color=Z,
                             scales={'column': sc_x, 'row': sc_y, 'color': sc_color},
                             opacity=0.4)

    # 4. MARKS
    start_pos = anim_data[0] 
    
    # Active Walkers
    walker_heads = bq.Scatter(x=start_pos[:, 0], y=start_pos[:, 1],
                              scales={'x': sc_x, 'y': sc_y},
                              colors=['#ff3300'], default_size=40, 
                              stroke='black', stroke_width=1)
    
    # Trails
    init_trail_x = np.column_stack([start_pos[:, 0], start_pos[:, 0]])
    init_trail_y = np.column_stack([start_pos[:, 1], start_pos[:, 1]])
    
    trails = bq.Lines(x=init_trail_x, y=init_trail_y,
                      scales={'x': sc_x, 'y': sc_y},
                      colors=['#00ccff'] * n_w, 
                      opacities=[0.6] * n_w, stroke_width=2)

    # Collected History (Persistent)
    collected_scatter = bq.Scatter(x=[], y=[], scales={'x': sc_x, 'y': sc_y},
                                   colors=['#000088'], default_size=10, default_opacities=[0.3])

    # Marginals
    # Top: Standard
    hist_x = bq.Hist(sample=[], scales={'sample': sc_x, 'count': sc_dens_x},
                     colors=['#444444'], opacities=[0.6], bins=40, normalized=True)
    
    # Right: Vertical orientation, mapped to Inverted Density Scale
    hist_y = bq.Hist(sample=[], scales={'sample': sc_y, 'count': sc_dens_y},
                     colors=['#444444'], opacities=[0.6], bins=40, normalized=True,
                     orientation='vertical')

    # 5. LAYOUT & ALIGNMENT (Pixel Perfect)
    # We lock the margins to ensure the plot areas align exactly
    # Top: 10, Bottom: 50 (Total 60 vertical loss)
    # Left: 50, Right: 10 (Total 60 horizontal loss)
    MARGINS_MAIN = {'top': 10, 'bottom': 50, 'left': 50, 'right': 10}
    MARGINS_TOP  = {'top': 10, 'bottom': 0,  'left': 50, 'right': 10} # Bottom 0 to touch main
    MARGINS_RIGHT= {'top': 10, 'bottom': 50, 'left': 0,  'right': 10} # Left 0 to touch main

    ax_x = bq.Axis(scale=sc_x, label='X', grid_lines='solid')
    ax_y = bq.Axis(scale=sc_y, label='Y', orientation='vertical', grid_lines='solid')
    
    # Figures
    fig_joint = bq.Figure(marks=[heatmap, collected_scatter, trails, walker_heads], 
                          axes=[ax_x, ax_y], 
                          layout=Layout(width='500px', height='500px'),
                          fig_margin=MARGINS_MAIN,
                          min_aspect_ratio=1, max_aspect_ratio=1)
    
    fig_marg_x = bq.Figure(marks=[hist_x], axes=[], 
                           layout=Layout(width='500px', height='100px'),
                           fig_margin=MARGINS_TOP)

    fig_marg_y = bq.Figure(marks=[hist_y], axes=[], 
                           layout=Layout(width='100px', height='500px'),
                           fig_margin=MARGINS_RIGHT)

    # Grid
    top_row = HBox([fig_marg_x, widgets.Box(layout=Layout(width='100px'))]) 
    bot_row = HBox([fig_joint, fig_marg_y])
    display(VBox([top_row, bot_row]))
    
    # 6. ANIMATION
    stride = 1 
    history_x, history_y = [], []
    steps = sampler.steps 
    
    for t in range(0, total_frames, stride):
        current_data = anim_data[t]
        
        # Trails
        start_trail = max(0, t - trail_len_param * 5)
        trail_slice = anim_data[start_trail : t+1] 
        if trail_slice.shape[0] < 2:
            trail_x = np.column_stack([current_data[:, 0], current_data[:, 0]])
            trail_y = np.column_stack([current_data[:, 1], current_data[:, 1]])
        else:
            trail_x = trail_slice[:, :, 0].T
            trail_y = trail_slice[:, :, 1].T
        
        with walker_heads.hold_sync():
            walker_heads.x = current_data[:, 0]
            walker_heads.y = current_data[:, 1]
            trails.x = trail_x
            trails.y = trail_y
            
            # Sampling Event: Updates History & Marginals
            if t > 0 and t % steps == 0:
                history_x.extend(current_data[:, 0])
                history_y.extend(current_data[:, 1])
                
                hist_x.sample = history_x
                hist_y.sample = history_y
                
                # Update persistent scatter
                if len(history_x) < 5000:
                    collected_scatter.x = history_x
                    collected_scatter.y = history_y
                else:
                    collected_scatter.x = history_x[-5000:]
                    collected_scatter.y = history_y[-5000:]
        
        time.sleep(speed_sld.value / 1000.0)

# Replace handler
def on_run(b):
    with output_area:
        output_area.clear_output()
        print("Strategy: Initializing...")
        try:
            U = create_jax_u(p_text.value, 2)
            sampler = HMCSampler(U=U)
            
            sampler.n_walkers = w_sld.value
            sampler.n_samples = s_sld.value
            sampler.n_burnin = b_sld.value
            sampler.lf_length = sz_sld.value
            sampler.steps = st_sld.value
            sampler.store_orbits = orb_cb.value 
            sampler.qi = np.zeros(2) 
            
            print("Strategy: Commencing HMC...")
            sampler.run_hmc()
            print("Strategy: Sampling Complete.")
            
            if orb_cb.value:
                run_fast_animation(sampler, U, trail_sld.value)
            else:
                sampler.plot_samples()
                plt.show()
                
        except Exception as e:
            import traceback
            traceback.print_exc()

run_btn.on_click(on_run)

def on_clear(b): output_area.clear_output()
clear_btn.on_click(on_clear)


# --- Final Grid ---
c1 = VBox([WidgetHTML(HDR.format(text="1. Potential")), p_text, d_drop], layout=COL_STYLE)
c2 = VBox([WidgetHTML(HDR.format(text="2. Logistics")), w_sld, s_sld, b_sld], layout=COL_STYLE)
c3 = VBox([WidgetHTML(HDR.format(text="3. Integrator")), sz_sld, st_sld, orb_cb, trail_sld, speed_sld], layout=COL_STYLE)

display(VBox([
    HBox([c1, c2, c3]), 
    HBox([run_btn, clear_btn, kill_btn_html], layout=Layout(margin='10px 5px')), 
    output_area, 
    shutdown_logic
]))